***

Preparing Workspace

***

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

if user == 'jfontes':
    # Git
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

***

Road_3

***

In [ ]:
indicator_name = 'Road_3'

file_name = f"{indicator_name} Pavement Conditions V2.xlsx"
df_counties = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data')

display(df_counties.head())

In [ ]:
# Set Indicator
indicator_name = 'Road_3'
plot_name = 'pavement_conditions'
export = False


## Importing ---

file_name = f"{indicator_name} Pavement Conditions V2.xlsx"
df_counties = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data')


## Organizing ---

df_plot = df_counties.copy()


df_plot = df_plot[df_plot['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]

df_plot = df_plot[['County', 'Lanemiles'] + sequence(2008, 2020, 2)]
df_plot = pd.melt(df_plot, id_vars = ['County', 'Lanemiles'], var_name = 'Year', value_name = 'PCI')

df_mpo = df_plot.copy()

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Lanemiles"]) # weighted average
df_mpo = df_mpo.groupby(['Year'], as_index = False).agg(PCI = ('PCI', wm))
df_mpo['PCI'] = round(df_mpo['PCI'], 0)
df_mpo['PCI'] = df_mpo['PCI'].astype(int)
df_mpo['County'] = 'SACOG (Average)'


display(df_plot.head(), df_mpo.head())



## Plotting ---

color_map = {
    'Sacramento': '#1F45FC'
    , 'Placer': '#1E90FF'
    , 'Yolo': '#9DC209'
    , 'El Dorado': '#FBB117'
    , 'Sutter': '#DC381F'
    , 'Yuba': '#7E587E'
}



fig = px.line(df_plot, x='Year', y='PCI', color='County', color_discrete_map=color_map, markers=True)

title = '<b>Local Street and Road Pavement Condition, by County</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=5, range = [49,81])
fig.update_traces(hovertemplate="%{y}")

fig.add_trace(go.Scatter(x=df_mpo["Year"], y=df_mpo['PCI']
                         , name = 'SACOG (Average)'
                         , line=go.scatter.Line(color="#2C3539", dash="dot")
                        ))



plot_agol(export=export)


***

Road_1

***

In [ ]:
indicator_name = 'Road_1'

file_name = f"{indicator_name} Gas Price download.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data_GasPrice', skiprows=1)

display(df_mpo.head())

In [ ]:
# Set Indicator
indicator_name = 'Road_1'
plot_name = 'gas_prices'
export = False


## Importing ---

file_name = f"{indicator_name} Gas Price download.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data_GasPrice', skiprows=1)



## Organizing ---
df_plot = df_mpo.copy()


df_plot = df_plot.drop('Unnamed: 0', axis = 1)
df_plot.columns = ['Date', 'avg_price']

df_plot['Date'] = pd.to_datetime(df_plot['Date'])
df_plot['Year'] = df_plot['Date'].dt.year


df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West monthly')
df_cpi['Date'] = pd.to_datetime(df_cpi['Month/Year Date'], format='%b-%Y')
df_cpi['Date'] = df_cpi['Date'].dt.to_period('M')

df_plot['Date'] = df_plot['Date'].dt.to_period('M')

df_cpi = df_cpi[['Date', 'CPI Adjusted (2022)']]
df_plot = df_plot.merge(df_cpi, on = 'Date', how = 'left')
df_plot['avg_price'] = round(df_plot['avg_price']/df_plot['CPI Adjusted (2022)'])
df_plot = df_plot.drop(['CPI Adjusted (2022)'], axis = 1)

df_plot = df_plot.groupby(['Year'], as_index=False).agg(avg_price = ('avg_price' ,'mean'))

df_plot['avg_price'] = round(df_plot['avg_price'], 1)

display(df_plot.head())



## Plotting ---


fig = px.bar(df_plot, x='Year', y='avg_price')
fig.update_traces(marker_color='#1E90FF')

title = '<b>Inflation-adjusted Average Price of Gasoline</b>  <br><sup>California</sup> '
fig.update_yaxes(tick0=0, dtick=1, tickprefix='$', range = [0, 6.1])
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(showlegend = False)


plot_agol(export=export)


***

Road_4

***

In [ ]:
indicator_name = 'Road_4'

file_name = f"{indicator_name} ZEV_Sales download.xlsx"
df_counties = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data_County')

display(df_counties.head())

In [ ]:
# Set Indicator
indicator_name = 'Road_4'
plot_name = 'EV_sales'
export = False


## Importing ---

file_name = f"{indicator_name} ZEV_Sales download.xlsx"
df_counties = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data_County')


## Organizing ---

df_plot = df_counties.copy()

df_plot = df_plot[df_plot['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
df_plot = df_plot[df_plot['Data Year'] == 2023]

df_plot = df_plot.groupby(['Data Year', 'County'], as_index = False)['Number of Vehicles'].sum()
df_mpo  = df_plot.groupby(['Data Year'          ], as_index = False)['Number of Vehicles'].sum()
df_mpo['County'] = 'Total'

df_plot = pd.concat([df_plot, df_mpo])


## Plotting ---

color_map = {
    'El Dorado':'#1E90FF'
   , 'Placer':'#1E90FF'
   , 'Sacramento':'#1E90FF'
   , 'Sutter':'#1E90FF'
   , 'Yolo':'#1E90FF'
   , 'Yuba':'#1E90FF'
    , 'Total':'#9DC209'
}


fig = px.bar(df_plot, x='County', y='Number of Vehicles'
            , color='County'
            , color_discrete_map=color_map)


title = '<b>Zero Emission Vehicle Sales, 2023</b>   <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=1000, range = [0, 5100], tickformat = ',.0f')
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(showlegend=False)


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator_name = 'Road_4'
plot_name = 'EV_sales_per_1k_pop'
export = False


## Importing ---
indicator_name = 'Road_4'

file_name = f"{indicator_name} ZEV_Sales download.xlsx"
df_counties = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data_County')


## Organizing ---

df_plot = df_counties.copy()
df_plot = df_plot[df_plot['Data Year'] == 2023]
df_plot = df_plot.rename(columns = {'Data Year':'Year'})

df_state  = df_plot.groupby(['Year'], as_index = False)['Number of Vehicles'].sum()
df_state['Geography'] = 'California'

df_mpo = df_plot[df_plot['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]

df_mpo  = df_mpo.groupby(['Year'], as_index = False)['Number of Vehicles'].sum()
df_mpo['Geography'] = 'SACOG'

df_pop = pd.read_excel(os.path.join(path_plots, 'Data', 'Pop_5 DOF MPO.xlsx'), sheet_name = 'Data')
df_pop = df_pop[df_pop['Year'] == 2023]
df_pop = df_pop[['MPO', 'Year', 'Population']]
df_pop = df_pop.rename(columns = {'MPO':'Geography'})

df_mpo = df_mpo.merge(df_pop[df_pop['Geography'] == 'SACOG'], on = ['Year', 'Geography'])

df_pop = df_pop.groupby(['Year'], as_index = False)['Population'].sum()
df_pop['Geography'] = 'California'

df_state = df_state.merge(df_pop, on = ['Year', 'Geography'])

df_plot = pd.concat([df_mpo, df_state])

df_plot['Sales/1k'] = round(df_plot['Number of Vehicles']/df_plot['Population']*1000, 1)

display(df_plot.head())



## Plotting ---

color_map = {
    'SACOG':'#9DC209'
    , 'California':'#1E90FF'
}

fig = px.bar(df_plot, y='Sales/1k', x='Year'
             , color='Geography'
             , barmode='group'
             , color_discrete_map=color_map)

title = '<b>Zero Emission Sales per 1,000 People, 2023</b>   <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=0.5, range = [0, 3.6])
fig.update_traces(hovertemplate='%{y}')


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator_name = 'Road_4'
plot_name = 'EV_sales'
export = False



## Importing ---

file_name = f"{indicator_name} ZEV_Sales download.xlsx"
df_counties = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Data_County')


## Organizing ---

df_plot = df_counties.copy()

df_plot = df_plot[df_plot['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]

df_plot = df_plot.groupby(['Data Year', 'County'], as_index = False)['Number of Vehicles'].sum()

df_plot = df_plot[df_plot['Data Year'] >= 2010]


df_plot['Sort'] = pd.Categorical(df_plot['County'], [
    'Sacramento'
    , 'Placer'
    , 'Yolo'
    , 'El Dorado'
    , 'Sutter'
    , 'Yuba'
])

df_plot = df_plot.sort_values(['Sort', 'Data Year'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())

## Plotting ---

color_map = {
    'Sacramento': '#1F45FC'
    , 'Placer': '#1E90FF'
    , 'Yolo': '#9DC209'
    , 'El Dorado': '#FBB117'
    , 'Sutter': '#DC381F'
    , 'Yuba': '#7E587E'
}




fig = px.bar(df_plot, x='Data Year', y='Number of Vehicles'
            , color='County'
            , color_discrete_map=color_map)


title = '<b>Zero Emission Vehicle Sales, 2023</b>   <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=2000, range = [0, 18100], tickformat = ',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)
